# Black-box current-evidence rerun (clean-only) — `blackbox_fullstate`

**Goal.** Regenerate a black-box / semi-structured baseline on the current-main
(`g3_5_7` mirror) regime so the chapter-section-8 "structured >> black-box" claim has a
clean-mirror anchor directly comparable to the four T2 models. Catalog-era black-box numbers
come from the 2026-04 `sweep_oc_core_default_*` suite — a different regime that collapsed
`phnode_full` on 3/6 clean seeds and flipped `phnode_qforce` clean 0.57<->3.76, so they
cannot be cross-compared with T2.

**Matrix (this notebook).** `blackbox_fullstate` x {clean} x seeds {42,43,44,45,46} = 5 training
runs (clean-only via `PHASE1A_PROTOCOLS=clean`), then the clean suite is evaluated across
{clean, nominal_eval, degraded_eval, heading_biased_eval} under iid_noisy_ic + v4_lite eval
protocols — identical to the eval the T2 clean suites received.

**Parallelism.** Run the three black-box notebooks (`blackbox_fullstate`,
`se3_momentum_blackbox`, `se3_accel_blackbox`) in separate Colab sessions. Distinct
`RUN_TAG`s, no collision.

**After all three finish:** sync `checkpoints/sweep_oc_phase1a_decision_clean_t2_wpfrag_*`
back, then locally rerun `scripts/export_section8_t2_evidence.py` — the black-box models are
already in its MODELS list, so the clean rows append to `aggregate.csv` automatically.

> Watch list: black-box models are collapse-prone (`blackbox_fullstate` was nan/86.8m in the
> old catalog). The anomaly-scan cell flags any clean-mirror collapse; a real collapse is a
> legitimate instability finding (handle per B1: exclude from aggregate, annotate).

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA: {torch.version.cuda}")
print(f"cuDNN: {torch.backends.cudnn.version()}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path(os.environ.get(
    "AUV_PROJECT_DIR",
    "/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7",
))
assert PROJECT_DIR.exists(), f"Project directory not found: {PROJECT_DIR}"
%cd $PROJECT_DIR

In [ ]:
%pip install -q torchdiffeq pandas

## 0. Configuration

In [ ]:
import os

# Runtime
os.environ["PYTHON_BIN"] = "python"
os.environ["DEVICE"] = "cuda"
os.environ["LOCAL_PROXY_ROOT"] = "/content/_proxy_suites"

# ---- identity (one black-box model per notebook) ----
os.environ["RUN_TAG"] = "t2_wpfrag_blackbox_fullstate"
os.environ["DATASET"] = "data/auv_oc_traj1000_blk150_s23_d0be9434.pkl"
os.environ["NOISE_REFERENCE"] = "remus100_dr"
os.environ["PHASE1A_LOG_DIR"] = str(PROJECT_DIR / "checkpoints" / "phase1a_logs" / os.environ["RUN_TAG"])
os.environ["PHASE1A_METADATA_DIR"] = str(PROJECT_DIR / "checkpoints" / f"phase1a_metadata_{os.environ['RUN_TAG']}")

# ---- matrix: single black-box model x CLEAN-ONLY x 5 seeds ----
os.environ["PHASE1A_MODELS"] = "blackbox_fullstate"
os.environ["SMOKE1_MODELS"] = "blackbox_fullstate"
os.environ["PHASE1A_PROTOCOLS"] = "clean"        # clean-only knob (driver default is "clean iid v4lite")
os.environ["SMOKE_SEEDS"] = "42"                 # cheap protocol-correctness gate (1 seed)
os.environ["DECISION_SEEDS"] = "42 43 44 45 46"  # full current-evidence seed set (matches T2)

# ---- Evaluation contract (identical to T2 clean suite) ----
os.environ["SMOKE_EVAL_NUM_TRAJ_PER_SCENARIO"] = "6"
os.environ["DECISION_EVAL_NUM_TRAJ_PER_SCENARIO"] = "30"
os.environ["EVAL_TIMES"] = "10 30 60"
os.environ["EVAL_SCENARIOS"] = "PRBS CHIRP OU"
os.environ["EVAL_BASE_SEED"] = "42"
os.environ["EVAL_NOISE_SEED"] = "2024"
os.environ["EVAL_PROGRESS_EVERY"] = "5"
os.environ["EVAL_NUM_DIAGNOSTIC_PLOTS"] = "6"
# Full robustness set: evaluate the clean-trained suite across all 4 profiles (caveat B).
os.environ["IID_EVAL_PROFILES"] = "clean nominal_eval degraded_eval heading_biased_eval"
os.environ["V4_EVAL_PROFILES"] = "nominal_eval"

# ---- Audit gate ----
os.environ["STRICT_ZERO_NOISE_AUDIT"] = "1"
os.environ["SOFT_MIN_EPOCH_SCALE"] = "0.05"

print("RUN_TAG          =", os.environ["RUN_TAG"])
print("PHASE1A_MODELS   =", os.environ["PHASE1A_MODELS"])
print("PHASE1A_PROTOCOLS=", os.environ["PHASE1A_PROTOCOLS"])
print("DECISION_SEEDS   =", os.environ["DECISION_SEEDS"])
print("IID_EVAL_PROFILES=", os.environ["IID_EVAL_PROFILES"])

## 1. Preflight
Confirms target suite/proxy dirs are absent and saves suite-level run config + environment metadata. If it fails, change `RUN_TAG` or remove the target dirs.

In [ ]:
os.environ["MODE"] = "preflight"
!bash scripts/run_phase1a_oc_v4lite.sh

## 2. Smoke gate (seed 42, clean only)
Cheap protocol-correctness check for the clean path + strict zero-noise audit. Smoke results are flow-validation only — never cited.

In [ ]:
os.environ["MODE"] = "smoke1_train"
!bash scripts/run_phase1a_oc_v4lite.sh

In [ ]:
os.environ["MODE"] = "smoke1_eval"
!bash scripts/run_phase1a_oc_v4lite.sh

## 3. Decision train (clean x 5 seeds)
The 5 evidence-bearing clean training runs for this black-box model.

In [ ]:
os.environ["MODE"] = "decision_train"
!bash scripts/run_phase1a_oc_v4lite.sh

## 4. Decision rollout eval (clean suite x 4 robustness profiles)
Evaluates the clean-trained suite across `clean nominal_eval degraded_eval heading_biased_eval` (iid eval protocol) plus the `v4_lite` eval at `nominal_eval`.

In [ ]:
os.environ["MODE"] = "decision_eval"
!bash scripts/run_phase1a_oc_v4lite.sh

## 5. Provenance injection (per-run `_audit_meta/`)
Records git HEAD + environment fingerprint into every decision run dir so these runs are distinguishable from catalog-era drift.

In [ ]:
# Inject per-run provenance. The training flow does NOT write these automatically,
# so for paper-grade evidence we record them here (see docs/provenance_audit_phnode_full_clean.md sec 5.2).
import os, sys, subprocess, datetime
from pathlib import Path
import torch

RUN_TAG = os.environ["RUN_TAG"]
ckpt = PROJECT_DIR / "checkpoints"
# clean-only sweep: only the clean suite exists, but we scan the full triple defensively.
suites = [ckpt / f"sweep_oc_phase1a_decision_{p}_{RUN_TAG}" for p in ("clean", "iid", "v4lite")]

def _sh(args):
    try:
        return subprocess.run(args, cwd=PROJECT_DIR, capture_output=True, text=True).stdout.strip()
    except Exception as exc:  # noqa: BLE001
        return f"<unavailable: {exc}>"

head = _sh(["git", "rev-parse", "HEAD"])
diffstat = _sh(["git", "diff", "HEAD", "--stat"])
env_txt = "\n".join([
    f"python={sys.version.split()[0]}",
    f"torch={torch.__version__}",
    f"cuda={torch.version.cuda}",
    f"cudnn={torch.backends.cudnn.version()}",
    f"gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}",
    f"captured_at={datetime.datetime.now().isoformat()}",
])

n = 0
for suite in suites:
    if not suite.exists():
        continue
    for cfg in suite.rglob("config.json"):
        run_dir = cfg.parent
        meta = run_dir / "_audit_meta"
        meta.mkdir(exist_ok=True)
        (meta / "code_revision.txt").write_text(f"git_head={head}\n\n{diffstat}\n")
        (meta / "environment.txt").write_text(env_txt + "\n")
        n += 1
print(f"provenance written for {n} run dirs")
print("git_head =", head)

## 6. Anomaly scan
Flags any run with the `no successful training batches` catastrophic-gradient signature. For black-box models a clean-mirror collapse is a legitimate instability result — flag and handle per B1 (exclude from the quantitative aggregate, annotate).

In [ ]:
# Quick anomaly scan: catch seed46/seed43-style catastrophic training before trusting numbers.
# Black-box models are collapse-prone (blackbox_fullstate collapsed in the 2026-04 catalog:
# s44=86.8m, s42/s43=nan), so a clean-mirror collapse here is a LEGITIMATE black-box instability
# finding, not an artifact -- still flag it and handle per the B1 policy (exclude from the
# quantitative aggregate, annotate transparently).
import os, re
from pathlib import Path

RUN_TAG = os.environ["RUN_TAG"]
ckpt = PROJECT_DIR / "checkpoints"
suites = [ckpt / f"sweep_oc_phase1a_decision_{p}_{RUN_TAG}" for p in ("clean", "iid", "v4lite")]

print(f"{'suite':<48}{'run':<32}{'best_epoch':>10}{'best_loss':>14}{'nbad':>6}  flag")
for suite in suites:
    if not suite.exists():
        continue
    for cfg in sorted(suite.rglob("config.json")):
        run_dir = cfg.parent
        log = run_dir / "training.log"
        text = log.read_text() if log.exists() else ""
        nbad = text.count("no successful training batches")
        best_epoch, best_loss = "?", "?"
        m = re.findall(r"[Bb]est.*?epoch[^0-9]*([0-9]+).*?([0-9]+\.[0-9eE+-]+)", text)
        if m:
            best_epoch, best_loss = m[-1]
        flag = "  <-- CHECK (possible artifact / instability)" if nbad > 0 else ""
        print(f"{suite.name[:46]:<48}{run_dir.name[:30]:<32}{str(best_epoch):>10}{str(best_loss):>14}{nbad:>6}{flag}")
print("\nFlagged seeds: handle per B1 (exclude from aggregate via train_anomaly, annotate).")

## 7. Next step (after all three black-box notebooks finish)
1. Ensure `checkpoints/sweep_oc_phase1a_decision_clean_t2_wpfrag_{blackbox_fullstate,se3_momentum_blackbox,se3_accel_blackbox}` are synced back.
2. Locally rerun `python scripts/export_section8_t2_evidence.py` (black-box models are already in MODELS).
3. The clean rows append to `analysis/section8_current_evidence/aggregate.csv` for the §8 structured-vs-black-box table.
4. No shared-catalog rebuild is needed — the export reads the decision suites directly.